# Mat & Comp – ML 7: Evaluación, entrenamiento/prueba y sobreajuste
## ¿Aprendió o memorizó?

**Curso:** Machine learning (Sección 3) · **Duración sugerida:** 2 sesiones de 50 minutos · **Requisito:** ML 6

### Objetivos de aprendizaje
Al terminar esta lección podrás:
1. Explicar el **sobreajuste** y el **subajuste** con ejemplos y gráficas.
2. Separar los datos en **entrenamiento** y **prueba** y medir el error con datos que el modelo no vio.
3. Usar la **validación cruzada** para elegir un modelo sin gastar los datos de prueba.
4. Leer una **matriz de confusión** y calcular **exactitud**, **precisión** y **sensibilidad**.
5. Reconocer por qué la exactitud sola puede engañar.

## 🔍 Dos estudiantes y un examen

Antes de un examen de matemáticas, el profesor entrega un taller de 20 ejercicios resueltos.

- **Ana** entiende cómo se resuelve cada ejercicio.
- **Beto** se aprende de memoria las 20 respuestas.

Si el examen trae **los mismos** 20 ejercicios, los dos sacan 5.0.

✏️ *¿Qué examen pondrías tú para saber quién aprendió de verdad? ¿Qué le pasaría a Beto?*

**Mi respuesta:**

Con los modelos de machine learning pasa exactamente lo mismo. En ML 4 y ML 5 vimos que un modelo puede acertar el 100 % de los datos con que se entrenó y fallar con datos nuevos. Hoy vamos a ordenar esas ideas y aprender a evaluar un modelo **sin hacernos trampa**.

## 1. Un experimento: la temperatura de un día

Un termómetro en Bogotá registra la temperatura 30 veces a lo largo de un día. La temperatura sube en la tarde y baja en la madrugada, pero cada medición tiene algo de **ruido** (una nube, una ráfaga de viento, el error del termómetro).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(3)
hora = np.sort(rng.uniform(0, 24, 30)).round(2)
temp = (14 + 5 * np.sin(2 * np.pi * (hora - 9) / 24) + rng.normal(0, 1.25, 30)).round(1)

plt.scatter(hora, temp, color="black", s=25)
plt.xlabel("hora del día")
plt.ylabel("temperatura (°C)")
plt.title("30 mediciones de temperatura")
plt.xticks(range(0, 25, 3))
plt.grid(alpha=0.3)
plt.show()

Vamos a ajustar **polinomios** a estos datos con `np.polyfit`, como hicimos con la recta en ML 2. Un polinomio de grado 1 es una recta, de grado 2 una parábola, y de grado alto una curva con muchas subidas y bajadas.

Para que los cálculos sean estables con grados altos, por dentro usamos la hora dividida por 24 (un número entre 0 y 1). Estas dos funciones se encargan de eso:

In [ ]:
def ajustar(x, y, grado):
    return np.polyfit(x / 24, y, grado)        # coeficientes del polinomio

def predecir(coef, x):
    return np.polyval(coef, x / 24)

def ecm(y_real, y_pred):                        # error cuadrático medio (ML 2)
    return np.mean((y_real - y_pred) ** 2)

xs = np.linspace(0, 24, 400)
fig, ejes = plt.subplots(1, 3, figsize=(16, 4.5), sharey=True)
for eje, grado in zip(ejes, [1, 3, 15]):
    coef = ajustar(hora, temp, grado)
    eje.scatter(hora, temp, color="black", s=20)
    eje.plot(xs, predecir(coef, xs), color="crimson")
    eje.set_ylim(5, 25)
    eje.set_title(f"grado {grado}: ECM = {ecm(temp, predecir(coef, hora)):.2f}")
    eje.set_xlabel("hora")
ejes[0].set_ylabel("temperatura (°C)")
plt.show()

- El **grado 1** (una recta) es demasiado simple: no puede subir y bajar. Esto se llama **subajuste**.
- El **grado 3** sigue la forma general.
- El **grado 15** tiene el **menor error** con estos datos… pero hace curvas extrañas para pasar cerca de cada punto, incluido el ruido. ¿Le creerías su predicción a las 23:30?

El error con los datos de entrenamiento **siempre baja** cuando el modelo es más complejo. Por eso no sirve para elegir el modelo: premia al que memoriza, como Beto.

## 2. Datos de entrenamiento y datos de prueba

La solución es la del examen con preguntas nuevas: **separar** los datos en dos partes antes de empezar.

- **Entrenamiento** (*train*): con estos datos el modelo aprende.
- **Prueba** (*test*): se guardan bajo llave y solo se usan para **evaluar**. El modelo nunca los ve mientras aprende.

Separamos al azar 20 mediciones para entrenar y 10 para probar. `rng.permutation(30)` revuelve los números del 0 al 29.

In [ ]:
orden = np.random.default_rng(0).permutation(30)
ent, pru = orden[:20], orden[20:]                    # posiciones de cada parte

print("Entrenamiento:", len(ent), "mediciones")
print("Prueba:       ", len(pru), "mediciones")

plt.scatter(hora[ent], temp[ent], color="black", s=25, label="entrenamiento")
plt.scatter(hora[pru], temp[pru], color="orange", s=60, marker="s", label="prueba")
plt.xlabel("hora del día")
plt.ylabel("temperatura (°C)")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

Ahora ajustamos cada polinomio **solo con los datos de entrenamiento** y medimos el error en las dos partes:

In [ ]:
grados = range(0, 13)
err_ent, err_pru = [], []
for g in grados:
    coef = ajustar(hora[ent], temp[ent], g)
    err_ent.append(ecm(temp[ent], predecir(coef, hora[ent])))
    err_pru.append(ecm(temp[pru], predecir(coef, hora[pru])))
    print(f"grado {g:2}:  error entrenamiento {err_ent[-1]:6.2f}   ·   error prueba {err_pru[-1]:9.2f}")

In [ ]:
plt.plot(list(grados), err_ent, "o-", label="entrenamiento")
plt.plot(list(grados), err_pru, "s-", label="prueba")
plt.xlabel("grado del polinomio (complejidad del modelo)")
plt.ylabel("error cuadrático medio")
plt.legend()
plt.grid(alpha=0.3)
plt.title("Subajuste, buen ajuste y sobreajuste")
plt.show()

Esta es **la gráfica más importante del machine learning**. Léela de izquierda a derecha:

| Zona | Error de entrenamiento | Error de prueba | Diagnóstico |
|---|---|---|---|
| Grados bajos | alto | alto | **Subajuste**: el modelo es demasiado simple |
| Grados medios | bajo | bajo | **Buen ajuste**: aprendió la regla general |
| Grados altos | muy bajo | sube otra vez | **Sobreajuste**: memorizó el ruido |

✏️ *¿Qué grado elegirías? ¿Por qué no el que tiene menor error de entrenamiento?*

## 3. Con scikit-learn: `train_test_split`

scikit-learn tiene una función que separa los datos al azar en una línea. Volvamos a los estudiantes y a la pregunta de ML 4 y ML 5: ¿aprobó?

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier

datos = pd.read_csv("https://raw.githubusercontent.com/emersonjleon/compumath/main/lecciones/seccion3_ml/datos/estudiantes.csv").dropna()
X = datos[["horas_estudio", "horas_sueno", "horas_redes"]].to_numpy()
y = np.where(datos["nota_matematicas"] >= 3.0, "aprobó", "no aprobó")

X_ent, X_pru, y_ent, y_pru = train_test_split(X, y, test_size=0.3, random_state=1)
print("Entrenamiento:", X_ent.shape, "  Prueba:", X_pru.shape)

- `test_size=0.3` guarda el 30 % para la prueba.
- `random_state` fija la forma de revolver, para que todos obtengamos la misma división.
- Devuelve **cuatro** cosas, siempre en este orden: X de entrenamiento, X de prueba, y de entrenamiento, y de prueba.

In [ ]:
for prof in [1, 2, 3, None]:
    arbol = DecisionTreeClassifier(max_depth=prof, random_state=0).fit(X_ent, y_ent)
    print(f"árbol max_depth={str(prof):4}:  entrenamiento {arbol.score(X_ent, y_ent):.0%}   ·   prueba {arbol.score(X_pru, y_pru):.0%}")

Otra vez: el árbol sin límite acierta todo en entrenamiento y menos en prueba.

### ⚠️ La suerte de la división

Con solo 18 estudiantes de prueba, cada uno vale más de 5 puntos porcentuales. Si cambiamos `random_state`, la nota del modelo cambia bastante:

In [ ]:
notas = []
for semilla in range(10):
    X_e, X_p, y_e, y_p = train_test_split(X, y, test_size=0.3, random_state=semilla)
    arbol = DecisionTreeClassifier(max_depth=2, random_state=0).fit(X_e, y_e)
    notas.append(arbol.score(X_p, y_p))
print("Exactitud en prueba con 10 divisiones distintas:", np.round(notas, 2))
print(f"Promedio {np.mean(notas):.2f}, de {min(notas):.2f} a {max(notas):.2f}")

## 4. Validación cruzada: usar todos los datos para evaluar

Con pocos datos, una sola división depende mucho de la suerte. La **validación cruzada** de *k* partes (*k-fold*) lo resuelve:

1. Dividir los datos de entrenamiento en k partes iguales (por ejemplo 5).
2. Entrenar con 4 partes y evaluar con la quinta.
3. Repetir 5 veces, cambiando la parte que evalúa.
4. Promediar los 5 resultados.

Así **cada dato se usa para evaluar exactamente una vez**. El «dejar uno afuera» de ML 4 y ML 5 es el caso extremo: tantas partes como datos.

In [ ]:
from sklearn.model_selection import cross_val_score

arbol = DecisionTreeClassifier(max_depth=2, random_state=0)
resultados = cross_val_score(arbol, X_ent, y_ent, cv=5)
print("Las 5 notas:", resultados.round(2))
print(f"Promedio: {resultados.mean():.2f}")

La validación cruzada sirve para **elegir el modelo**: comparamos opciones usando solo los datos de entrenamiento, y dejamos los de prueba intactos.

Para k-NN recordemos la lección de ML 4: hay que **normalizar**. `make_pipeline(StandardScaler(), KNeighborsClassifier(...))` junta los dos pasos en un solo modelo: primero normaliza (`StandardScaler` resta el promedio y divide por la desviación) y luego clasifica. Además, en cada parte de la validación cruzada calcula el promedio y la desviación **solo** con los datos de entrenamiento de esa parte, como debe ser (verás por qué en los errores comunes).

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

print("Árboles:")
for prof in [1, 2, 3, 4, None]:
    r = cross_val_score(DecisionTreeClassifier(max_depth=prof, random_state=0), X_ent, y_ent, cv=5)
    print(f"  max_depth={str(prof):4}  validación cruzada {r.mean():.3f}")
print("k vecinos (normalizando):")
for k in [1, 3, 5, 9, 15]:
    modelo = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=k))
    r = cross_val_score(modelo, X_ent, y_ent, cv=5)
    print(f"  k={k:2}           validación cruzada {r.mean():.3f}")

### 🔑 La regla de oro

1. **Separa** los datos de prueba al principio y no los mires.
2. **Elige** el modelo y sus parámetros (`max_depth`, `k`, el grado…) con validación cruzada sobre los datos de entrenamiento.
3. **Evalúa una sola vez** el modelo elegido con los datos de prueba. Esa es su nota honesta.

Si usas los datos de prueba para elegir, se convierten en un taller más: el modelo (¡y tú!) termina «memorizando» también el examen.

In [ ]:
# La validación cruzada eligió k-NN normalizado; entre k = 3 y k = 5 (empate) tomamos k = 5
elegido = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5)).fit(X_ent, y_ent)
print(f"Nota final en prueba: {elegido.score(X_pru, y_pru):.0%}")

## 5. Más allá del porcentaje: la matriz de confusión

Hasta ahora medimos el porcentaje de aciertos. Su nombre técnico es **exactitud** (*accuracy*).

💡 **Una aclaración de nombres:** en ML 4 y ML 5 lo llamamos «precisión», como se dice en el lenguaje de todos los días. En machine learning, **precisión** es otra medida, que veremos en un momento. De aquí en adelante diremos **exactitud** para el porcentaje de aciertos.

La **matriz de confusión** cuenta **qué tipo** de errores comete el modelo: las filas son la respuesta real y las columnas la predicción.

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

pred = elegido.predict(X_pru)
etiquetas = ["aprobó", "no aprobó"]
M = confusion_matrix(y_pru, pred, labels=etiquetas)
print(M)

ConfusionMatrixDisplay(M, display_labels=etiquetas).plot(cmap="Blues", colorbar=False)
plt.xlabel("predicción del modelo")
plt.ylabel("respuesta real")
plt.show()

En la diagonal están los aciertos; fuera de ella, los errores. Hay dos tipos de error distintos: decir «aprobó» a quien no aprobó, y decir «no aprobó» a quien sí aprobó. ¿Cuál es más grave? Depende de para qué se use el modelo.

### Un detector que nunca detecta

Imagina una prueba para una enfermedad rara: de 1000 personas, solo 20 están enfermas. Un «detector» perezoso dice **siempre** «sano».

In [ ]:
real = np.array(["enfermo"] * 20 + ["sano"] * 980)
detector = np.array(["sano"] * 1000)

print(f"Exactitud: {np.mean(real == detector):.0%}")
print(confusion_matrix(real, detector, labels=["enfermo", "sano"]))

¡98 % de exactitud sin detectar a **ningún** enfermo! Cuando una clase es mucho más común que la otra, la exactitud engaña. Por eso se usan dos medidas más, centradas en la clase que nos importa (aquí, «enfermo»):

- **Sensibilidad** (*recall*): de los que **realmente** son enfermos, ¿a cuántos detectó? $\;\dfrac{\text{enfermos detectados}}{\text{enfermos reales}}$
- **Precisión** (*precision*): de los que el modelo **dijo** que son enfermos, ¿cuántos lo eran? $\;\dfrac{\text{enfermos detectados}}{\text{todos los que dijo enfermos}}$

El detector perezoso tiene sensibilidad 0 %: es inútil, aunque tenga 98 % de exactitud.

In [ ]:
from sklearn.metrics import precision_score, recall_score

# Un detector un poco mejor: detecta a 15 de los 20 enfermos, pero da 30 falsas alarmas
detector2 = np.array(["enfermo"] * 15 + ["sano"] * 5 + ["enfermo"] * 30 + ["sano"] * 950)
print(confusion_matrix(real, detector2, labels=["enfermo", "sano"]))
print(f"Exactitud:    {np.mean(real == detector2):.1%}")
print(f"Sensibilidad: {recall_score(real, detector2, pos_label='enfermo'):.1%}")
print(f"Precisión:    {precision_score(real, detector2, pos_label='enfermo'):.1%}")

Este detector tiene **menos** exactitud que el perezoso (96.5 %), ¡pero encuentra al 75 % de los enfermos! Su precisión es baja (1 de cada 3 alarmas es real), lo cual puede estar bien si después se confirma con un examen más completo.

✏️ *Para un detector de incendios, ¿qué es más importante: alta sensibilidad o alta precisión? ¿Y para un filtro que borra correos de spam?*

## 6. Errores comunes

**Error 1: evaluar con los datos de entrenamiento.** Es el error de Beto. Siempre da números demasiado buenos.

**Error 2: usar la prueba para elegir.** Si pruebas 50 modelos con los datos de prueba y te quedas con el mejor, ese número ya no es honesto. Elige con validación cruzada.

**Error 3: «fugas» de información.** Si normalizas (como en ML 4) usando el promedio de **todos** los datos antes de separar, los datos de prueba ya «se colaron» en el entrenamiento. Calcula el promedio y la desviación solo con los datos de entrenamiento.

**Error 4: X e y desalineados.** `X` e `y` deben tener la misma cantidad de filas, en el mismo orden:

In [ ]:
train_test_split(X, y[:-1], test_size=0.3)

## 🤔 ¿Qué pasa si…?

Usa la celda de abajo. Antes de ejecutar, **predice** el resultado.

1. …en la sección 2 usas grado 0? ¿Qué forma tiene la «curva» y qué predice?
2. …usas `test_size=0.5`? ¿Y `test_size=0.05`? ¿Qué se gana y qué se pierde?
3. …calculas `cross_val_score(KNeighborsClassifier(n_neighbors=1), X_ent, y_ent, cv=5)` y lo comparas con `KNeighborsClassifier(n_neighbors=1).fit(X_ent, y_ent).score(X_ent, y_ent)`?
4. …usas `cv=len(X_ent)` en `cross_val_score`? ¿A qué método de ML 4 corresponde?
5. …el detector perezoso dice siempre «enfermo»? ¿Cuánto valen su exactitud, su sensibilidad y su precisión?

In [ ]:
# Prueba aquí tus ideas

## ✍️ Ejercicios

**Ejercicio 1 (fácil). A mano.** Un modelo que detecta mangos dañados en una planta empacadora tiene esta matriz de confusión (filas: real; columnas: predicción):

|  | predijo dañado | predijo bueno |
|---|---|---|
| **realmente dañado** | 40 | 10 |
| **realmente bueno** | 20 | 430 |

Calcula con lápiz la exactitud, la sensibilidad y la precisión (para la clase «dañado»). Compruébalo con numpy.

**Ejercicio 2 (fácil). La suerte de la división.** Repite la sección 2 con 10 divisiones distintas (semillas 0 a 9) y calcula, para los grados 3 y 9, el promedio del error de prueba. ¿Cuál es más confiable? ¿Cuál cambia más de una división a otra?

**Ejercicio 3 (medio). Elegir k como se debe.** Con los estudiantes, usa validación cruzada (`cv=5`) sobre los datos de entrenamiento para elegir el mejor `n_neighbors` entre 1 y 25 (solo impares). Grafica el resultado. Luego evalúa **una sola vez** el k elegido con los datos de prueba.

**Ejercicio 4 (medio). ¿Qué dígitos se confunden?** Con los dígitos escritos a mano (`load_digits`), separa 30 % para prueba con `random_state=0`, entrena `KNeighborsClassifier(n_neighbors=3)` y dibuja la matriz de confusión de 10 × 10. ¿Qué pares de dígitos se confunden más?

**Ejercicio 5 (medio). ¿Más datos ayudan?** Con los datos de la temperatura, simula más mediciones: genera 300 en lugar de 30 (con la misma fórmula y otra semilla), separa 2/3 para entrenar y 1/3 para probar, y vuelve a graficar el error de entrenamiento y de prueba por grado (de 0 a 12). ¿Qué le pasó al sobreajuste?

## 🏆 Reto opcional: tu propia validación cruzada

Programa la validación cruzada de 5 partes **sin** `cross_val_score`:

1. Revuelve las posiciones de los datos de entrenamiento con `np.random.default_rng(0).permutation(...)` y pártelas en 5 grupos con `np.array_split`.
2. Para cada grupo: entrena un árbol con los otros 4 (pista: `np.concatenate`) y evalúa con ese grupo.
3. Promedia. ¿Obtienes un resultado parecido al de `cross_val_score`? ¿Por qué podría no ser idéntico?
4. Extra: usa tu función para elegir el grado del polinomio de la temperatura, con el error cuadrático en lugar de la exactitud.

## 🎬 Cierre

Hoy aprendiste que:
- El error con los datos de entrenamiento **no** mide si el modelo aprendió: premia la memoria.
- Se separan datos de **entrenamiento** y de **prueba**; la prueba se usa **una sola vez**, al final.
- La **validación cruzada** permite comparar y elegir modelos sin tocar la prueba.
- **Subajuste**: modelo demasiado simple. **Sobreajuste**: modelo que memoriza el ruido.
- La **matriz de confusión** muestra qué errores comete un modelo; la **exactitud** puede engañar, y la **sensibilidad** y la **precisión** dicen más cuando una clase es rara.

**Para pensar:** ya tienes las herramientas para entrenar y evaluar modelos con honestidad. En la próxima lección veremos la **regresión logística**, un modelo que no solo clasifica, sino que dice **qué tan seguro** está de cada respuesta. 📈